In [1]:
import numpy as np
import pandas as pd
import seaborn as sb
import re
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (16, 9)
plt.style.use('ggplot')

from sklearn import tree
from sklearn.model_selection import KFold
from sklearn.tree import export_graphviz
from graphviz import Source
from IPython.display import display, Markdown, Image as PImage
from subprocess import check_call
import ipywidgets as widgets
import os

# --- Cargar datos ---
artists_billboard = pd.read_csv("./artists_billboard.csv")

# --- Preparar variables y datos para gráficos y análisis (con tu código resumido y corregido) ---
# (Aquí puedes pegar todo tu código de procesamiento, encoding, etc. sin repeticiones)

def edad_fix(anio):
    if anio==0:
        return None
    return anio

artists_billboard['anioNacimiento'] = artists_billboard.apply(lambda x: edad_fix(x['anioNacimiento']), axis=1)

def calcula_edad(anio, cuando):
    cad = str(cuando)
    momento = cad[:4]
    if anio == 0 or pd.isnull(anio):
        return None
    return int(momento) - anio

artists_billboard['edad_en_billboard'] = artists_billboard.apply(lambda x: calcula_edad(x['anioNacimiento'], x['chart_date']), axis=1)

age_avg = artists_billboard['edad_en_billboard'].mean()
age_std = artists_billboard['edad_en_billboard'].std()
age_null_count = artists_billboard['edad_en_billboard'].isnull().sum()
age_null_random_list = np.random.randint(int(age_avg - age_std), int(age_avg + age_std), size=age_null_count)
conValoresNulos = artists_billboard['edad_en_billboard'].isnull()

artists_billboard.loc[artists_billboard['edad_en_billboard'].isnull(), 'edad_en_billboard'] = age_null_random_list
artists_billboard['edad_en_billboard'] = artists_billboard['edad_en_billboard'].astype(int)

# Encodings
artists_billboard['moodEncoded'] = artists_billboard['mood'].map( {'Energizing': 6, 'Empowering': 6, 'Cool': 5, 'Yearning': 4, 'Excited': 5,
 'Defiant': 3, 'Sensual': 2, 'Gritty': 3, 'Sophisticated': 4, 'Aggressive': 4, 'Fiery': 4, 'Urgent': 3,
 'Rowdy': 4, 'Sentimental': 4, 'Easygoing': 1, 'Melancholy': 4, 'Romantic': 2, 'Peaceful': 1,
 'Brooding': 4, 'Upbeat': 5, 'Stirring': 5, 'Lively': 5, 'Other': 0, '': 0}).astype(int)

artists_billboard['tempoEncoded'] = artists_billboard['tempo'].map({'Fast Tempo': 0, 'Medium Tempo': 2, 'Slow Tempo': 1, '': 0}).astype(int)
artists_billboard['genreEncoded'] = artists_billboard['genre'].map({'Urban': 4, 'Pop': 3, 'Traditional': 2, 'Alternative & Punk': 1,
 'Electronica': 1, 'Rock': 1, 'Soundtrack': 0, 'Jazz': 0, 'Other':0, '':0}).astype(int)
artists_billboard['artist_typeEncoded'] = artists_billboard['artist_type'].map({'Female': 2, 'Male': 3, 'Mixed': 1, '': 0}).astype(int)

# Edad encoded
artists_billboard.loc[artists_billboard['edad_en_billboard'] <= 21, 'edadEncoded'] = 0
artists_billboard.loc[(artists_billboard['edad_en_billboard'] > 21) & (artists_billboard['edad_en_billboard'] <= 26), 'edadEncoded'] = 1
artists_billboard.loc[(artists_billboard['edad_en_billboard'] > 26) & (artists_billboard['edad_en_billboard'] <= 30), 'edadEncoded'] = 2
artists_billboard.loc[(artists_billboard['edad_en_billboard'] > 30) & (artists_billboard['edad_en_billboard'] <= 40), 'edadEncoded'] = 3
artists_billboard.loc[artists_billboard['edad_en_billboard'] > 40, 'edadEncoded'] = 4

# Duración encoded
artists_billboard.loc[artists_billboard['durationSeg'] <= 150, 'durationEncoded'] = 0
artists_billboard.loc[(artists_billboard['durationSeg'] > 150) & (artists_billboard['durationSeg'] <= 180), 'durationEncoded'] = 1
artists_billboard.loc[(artists_billboard['durationSeg'] > 180) & (artists_billboard['durationSeg'] <= 210), 'durationEncoded'] = 2
artists_billboard.loc[(artists_billboard['durationSeg'] > 210) & (artists_billboard['durationSeg'] <= 240), 'durationEncoded'] = 3
artists_billboard.loc[(artists_billboard['durationSeg'] > 240) & (artists_billboard['durationSeg'] <= 270), 'durationEncoded'] = 4
artists_billboard.loc[(artists_billboard['durationSeg'] > 270) & (artists_billboard['durationSeg'] <= 300), 'durationEncoded'] = 5
artists_billboard.loc[artists_billboard['durationSeg'] > 300, 'durationEncoded'] = 6

drop_elements = ['id','title','artist','mood','tempo','genre','artist_type','chart_date','anioNacimiento','durationSeg','edad_en_billboard']
artists_encoded = artists_billboard.drop(drop_elements, axis=1)

# --- Funciones para mostrar gráficas (usar plt.show() para mostrar inline en notebook) ---
def graficas_countplots():
    sb.countplot(data=artists_billboard, x="top", hue="top", palette="Set1", legend=False)
    plt.title("Count of 'top' variable")
    plt.show()

    sb.catplot(x='artist_type', data=artists_billboard, kind="count")
    plt.show()

    sb.catplot(x='mood', data=artists_billboard, kind="count", aspect=3)
    plt.xticks(rotation=45)
    plt.show()

    sb.catplot(x='tempo', data=artists_billboard, hue='top', kind="count")
    plt.show()

    sb.catplot(x='genre', data=artists_billboard, kind="count", aspect=3)
    plt.xticks(rotation=45)
    plt.show()

    sb.catplot(x='anioNacimiento', data=artists_billboard, kind="count", aspect=3)
    plt.xticks(rotation=45)
    plt.show()

def graficas_scatter():
    f1 = artists_billboard['chart_date'].values
    f2 = artists_billboard['durationSeg'].values
    colores = ['orange', 'blue']
    tamanios = [60, 40]

    asignar = []
    asignar2 = []
    for index, row in artists_billboard.iterrows():
        asignar.append(colores[row['top']])
        asignar2.append(tamanios[row['top']])

    plt.scatter(f1, f2, c=asignar, s=asignar2)
    plt.axis([20030101, 20160101, 0, 600])
    plt.title("Scatter plot: chart_date vs durationSeg colored by 'top'")
    plt.show()

    f1 = artists_billboard['edad_en_billboard'].values
    f2 = artists_billboard.index

    asignar = []
    conValoresNulos = artists_billboard['edad_en_billboard'].isnull()
    colores = ['orange', 'blue', 'green']

    for index, row in artists_billboard.iterrows():
        if conValoresNulos[index]:
            asignar.append(colores[2])
        else:
            asignar.append(colores[row['top']])

    plt.scatter(f1, f2, c=asignar, s=30)
    plt.axis([15, 50, 0, 650])
    plt.title("Scatter plot: edad_en_billboard vs index colored by 'top' and nulls")
    plt.show()

def mostrar_tablas_resumen():
    display(Markdown("### 🎵 Mood vs Top"))
    display(artists_encoded[['moodEncoded', 'top']].groupby(['moodEncoded'], as_index=False).agg(['mean', 'count', 'sum']))

    display(Markdown("### 👩‍🎤 Artist Type vs Top"))
    display(artists_encoded[['artist_typeEncoded', 'top']].groupby(['artist_typeEncoded'], as_index=False).agg(['mean', 'count', 'sum']))

    display(Markdown("### 🎧 Genre vs Top"))
    display(artists_encoded[['genreEncoded', 'top']].groupby(['genreEncoded'], as_index=False).agg(['mean', 'count', 'sum']))

    display(Markdown("### ⏱ Tempo vs Top"))
    display(artists_encoded[['tempoEncoded', 'top']].groupby(['tempoEncoded'], as_index=False).agg(['mean', 'count', 'sum']))

    display(Markdown("### ⌛ Duration vs Top"))
    display(artists_encoded[['durationEncoded', 'top']].groupby(['durationEncoded'], as_index=False).agg(['mean', 'count', 'sum']))

    display(Markdown("### 👶 Edad vs Top"))
    display(artists_encoded[['edadEncoded', 'top']].groupby(['edadEncoded'], as_index=False).agg(['mean', 'count', 'sum']))

def entrenar_y_mostrar_arbol():
    cv = KFold(n_splits=10)
    accuracies = list()
    max_attributes = len(list(artists_encoded.columns)) - 1  # quitar 'top'
    depth_range = range(1, max_attributes + 1)

    for depth in depth_range:
        fold_accuracy = []
        tree_model = tree.DecisionTreeClassifier(criterion='entropy',
                                                 min_samples_split=20,
                                                 min_samples_leaf=5,
                                                 max_depth=depth,
                                                 class_weight={1:3.5})
        for train_fold, valid_fold in cv.split(artists_encoded):
            f_train = artists_encoded.loc[train_fold]
            f_valid = artists_encoded.loc[valid_fold]

            model = tree_model.fit(X=f_train.drop(['top'], axis=1),
                                   y=f_train["top"])
            valid_acc = model.score(X=f_valid.drop(['top'], axis=1),
                                   y=f_valid["top"])
            fold_accuracy.append(valid_acc)

        avg = sum(fold_accuracy) / len(fold_accuracy)
        accuracies.append(avg)

    df_acc = pd.DataFrame({"Max Depth": depth_range, "Average Accuracy": accuracies})
    display(Markdown("### Resultados Accuracy por Max Depth"))
    display(df_acc)

    # Entrenar árbol final con profundidad 4
    y_train = artists_encoded['top']
    x_train = artists_encoded.drop(['top'], axis=1)

    decision_tree = tree.DecisionTreeClassifier(criterion='entropy',
                                                min_samples_split=20,
                                                min_samples_leaf=5,
                                                max_depth=4,
                                                class_weight={1:3.5})
    decision_tree.fit(x_train, y_train)

    # Exportar y generar imagen del árbol
    dot_path = "tree1.dot"
    png_path = "tree1.png"
    with open(dot_path, 'w') as f:
        f = export_graphviz(decision_tree,
                            out_file=f,
                            max_depth=7,
                            impurity=True,
                            feature_names=list(artists_encoded.drop(['top'], axis=1).columns),
                            class_names=['No', 'N1 Billboard'],
                            rounded=True,
                            filled=True)

    # Convertir .dot a .png
    check_call(['dot', '-Tpng', dot_path, '-o', png_path])

    display(Markdown(f"### Árbol de decisión (precisión entrenamiento: {round(decision_tree.score(x_train, y_train)*100,2)}%)"))
    display(PImage(png_path))

    return decision_tree

def mostrar_predicciones_y_caminos(decision_tree):
    # Datos para prueba
    x_test_havana = pd.DataFrame(columns=('top', 'moodEncoded', 'tempoEncoded', 'genreEncoded',
                                          'artist_typeEncoded', 'edadEncoded', 'durationEncoded'))
    x_test_havana.loc[0] = (1, 5, 2, 4, 1, 0, 3)

    x_test_believer = pd.DataFrame(columns=('top', 'moodEncoded', 'tempoEncoded', 'genreEncoded',
                                            'artist_typeEncoded', 'edadEncoded', 'durationEncoded'))
    x_test_believer.loc[0] = (0, 4, 2, 1, 3, 2, 3)

    # Predicciones
    y_pred_havana = decision_tree.predict(x_test_havana.drop(['top'], axis=1))
    y_proba_havana = decision_tree.predict_proba(x_test_havana.drop(['top'], axis=1))

    y_pred_believer = decision_tree.predict(x_test_believer.drop(['top'], axis=1))
    y_proba_believer = decision_tree.predict_proba(x_test_believer.drop(['top'], axis=1))

    print(f"Havana - Predicción: {y_pred_havana[0]} (Probabilidad: {round(y_proba_havana[0][y_pred_havana[0]]*100,2)}%)")
    print(f"Believer - Predicción: {'Sí llegó al Top 1' if y_pred_believer[0]==1 else 'No llegó al Top 1'} (Probabilidad: {round(y_proba_believer[0][y_pred_believer[0]]*100,2)}%)")
    print(f"Probabilidades Believer completas → No Top: {round(y_proba_believer[0][0]*100,2)}%, Top: {round(y_proba_believer[0][1]*100,2)}%")

    # Caminos de decisión
    havana_nodes = decision_tree.decision_path(x_test_havana.drop(['top'], axis=1)).indices
    believer_nodes = decision_tree.decision_path(x_test_believer.drop(['top'], axis=1)).indices

    dot_base = export_graphviz(
        decision_tree,
        out_file=None,
        feature_names=artists_encoded.drop(['top'], axis=1).columns,
        class_names=["No", "N1 Billboard"],
        rounded=True,
        filled=True,
        node_ids=True
    )

    def resaltar_varios_caminos(dot_text, caminos_colores):
        new_lines = []
        for line in dot_text.split('\n'):
            original_line = line
            for nodos, color in caminos_colores:
                for nodo in nodos:
                    if line.strip().startswith(f'{nodo} ['):
                        if 'fillcolor=' in line:
                            line = re.sub(r'fillcolor="[^"]+"', f'fillcolor="{color}"', line)
                        else:
                            line = line.replace(']', f', style=filled, fillcolor="{color}"]')
                        break
            new_lines.append(line)
        return '\n'.join(new_lines)

    coloreado = resaltar_varios_caminos(dot_base, [(havana_nodes, "red"), (believer_nodes, "deeppink")])

    graph = Source(coloreado)
    graph_path = "camino_havana_believer.png"
    graph.render("camino_havana_believer", format="png", cleanup=True)

    display(Markdown("### Árbol de decisión con caminos resaltados"))
    display(PImage(graph_path))

# --- Interfaz con ipywidgets ---

# Crear pestañas para organizar la presentación
tab = widgets.Tab()

# Crear cajas para cada sección
box_datos = widgets.Output()
box_graficos = widgets.Output()
box_tablas = widgets.Output()
box_arbol = widgets.Output()
box_predicciones = widgets.Output()

tab.children = [box_datos, box_graficos, box_tablas, box_arbol, box_predicciones]
tab.set_title(0, "Datos")
tab.set_title(1, "Gráficos")
tab.set_title(2, "Tablas Resumen")
tab.set_title(3, "Árbol Decisión")
tab.set_title(4, "Predicciones")

# Rellenar contenido de cada caja
with box_datos:
    display(Markdown("## Datos originales y resumen"))
    display(Markdown(f"**Shape:** {artists_billboard.shape}"))
    display(Markdown("### Primeras filas"))
    display(artists_billboard.head())
    display(Markdown("### Distribución de 'top'"))
    display(artists_billboard.groupby('top').size())

with box_graficos:
    graficas_countplots()
    graficas_scatter()

with box_tablas:
    mostrar_tablas_resumen()

with box_arbol:
    dtree = entrenar_y_mostrar_arbol()

with box_predicciones:
    mostrar_predicciones_y_caminos(dtree)

display(tab)
